# Clase 1 — Repaso de probabilidad para aprendizaje automático

**Redes Neuronales**

El objetivo no es hacer teoría abstracta, sino recuperar las ideas de probabilidad que vamos a usar durante la cursada y conectarlas con datos y código.

## Objetivos

Al terminar esta notebook deberías poder:

- interpretar una probabilidad como frecuencia relativa e incertidumbre,
- distinguir entre probabilidad marginal, conjunta y condicional,
- entender por qué $P(A\mid B)$ no es lo mismo que $P(B\mid A)$,
- aplicar la regla del producto y el teorema de Bayes,
- vincular estas ideas con problemas de clasificación en machine learning,
- usar ejemplos simples con `numpy`, `pandas` y `matplotlib`.

## 0. Librerías

Vamos a usar:

- **NumPy** para arreglos y simulaciones simples,
- **pandas** para tablas y conteos,
- **matplotlib** para gráficos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

rng = np.random.default_rng(42)

## 1. ¿Qué es una probabilidad?

Una probabilidad mide cuán plausible es que ocurra un evento.

Siempre vale:

$$
0 \leq P(A) \leq 1
$$

- $P(A)=0$: el evento es imposible.
- $P(A)=1$: el evento es seguro.
- valores intermedios indican distintos niveles de incertidumbre.

### Ejemplo simple
Si tiramos un dado equilibrado:

- $P(\text{sale 1}) = 1/6$
- $P(\text{sale par}) = 3/6 = 1/2$

En muchos problemas reales, una probabilidad también puede interpretarse como **frecuencia relativa**: si repetimos muchas veces un experimento, la proporción observada tiende a estabilizarse.

### Ejemplo 1 — Simulación de un dado con NumPy

In [1]:
# Simulamos 1000 lanzamientos de un dado
lanzamientos = rng.integers(1, 7, size=1000)

# Estimamos la probabilidad de obtener un número par
prob_par_estimada = np.mean(lanzamientos % 2 == 0)
prob_mayor_4_estimada = np.mean(lanzamientos > 4)

print("Probabilidad estimada de par:", prob_par_estimada)
print("Probabilidad estimada de ser mayor que 4:", prob_mayor_4_estimada)

NameError: name 'rng' is not defined

In [ ]:
# Conteo de frecuencias con pandas
serie_dado = pd.Series(lanzamientos, name="resultado")
frecuencias = serie_dado.value_counts().sort_index()
tabla_frecuencias = pd.DataFrame({
    "frecuencia": frecuencias,
    "frecuencia_relativa": frecuencias / frecuencias.sum()
})
tabla_frecuencias

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(tabla_frecuencias.index.astype(str), tabla_frecuencias["frecuencia_relativa"])
plt.xlabel("Cara del dado")
plt.ylabel("Frecuencia relativa")
plt.title("Frecuencias relativas en 1000 lanzamientos")
plt.show()

### Observación
Teóricamente cada cara tiene probabilidad $1/6 \approx 0.167$.  
En la simulación, las frecuencias relativas no dan exactamente eso, pero se acercan.

Esto ya conecta una idea importante: en datos reales, muchas veces **estimamos probabilidades a partir de frecuencias observadas**.

## 2. Eventos y variables

Un **evento** es algo que puede ocurrir.  
Ejemplos:

- “el dado sale par”,
- “un email es spam”,
- “una imagen contiene un gato”.

Una **variable aleatoria** es una forma de representar numérica o categóricamente el resultado de un fenómeno incierto.

En machine learning suele usarse esta notación:

- $X$: entrada o conjunto de atributos,
- $Y$: variable objetivo o etiqueta.

Por ejemplo:

- $X$: palabras presentes en un email,
- $Y$: `spam` o `no spam`.

Más adelante vamos a usar expresiones como:

$$
P(Y=\text{spam})
$$
y
$$
P(Y=\text{spam} \mid X=x)
$$

Para que eso tenga sentido, primero hay que entender marginales, conjuntas y condicionales.

## 3. Probabilidad marginal y probabilidad conjunta

Vamos a trabajar con un ejemplo simple de emails.

Supongamos que observamos 100 emails y registramos dos variables:

- si el email contiene la palabra **"gratis"**,
- si el email es **spam**.

In [ ]:
datos_emails = pd.DataFrame({
    "contiene_gratis": ["Sí"] * 30 + ["No"] * 70,
    "spam": ["Sí"] * 24 + ["No"] * 6 + ["Sí"] * 16 + ["No"] * 54
})

datos_emails.head()

In [ ]:
tabla = pd.crosstab(
    datos_emails["contiene_gratis"],
    datos_emails["spam"],
    margins=True
)
tabla

La tabla resume los 100 casos:

- 24 emails contienen “gratis” y son spam,
- 6 contienen “gratis” y no son spam,
- 16 no contienen “gratis” y son spam,
- 54 no contienen “gratis” y no son spam.

### Probabilidad marginal

La **probabilidad marginal** mira una variable sin condicionar por otra.

Por ejemplo:

$$
P(\text{Spam}) = 40/100 = 0.4
$$

$$
P(\text{Gratis}) = 30/100 = 0.3
$$

In [ ]:
p_spam = np.mean(datos_emails["spam"] == "Sí")
p_gratis = np.mean(datos_emails["contiene_gratis"] == "Sí")

print("P(Spam) =", p_spam)
print("P(Gratis) =", p_gratis)

### Probabilidad conjunta

La **probabilidad conjunta** mira que ocurran dos cosas al mismo tiempo.

Por ejemplo:

$$
P(\text{Gratis} \cap \text{Spam}) = 24/100 = 0.24
$$

In [ ]:
p_gratis_y_spam = np.mean(
    (datos_emails["contiene_gratis"] == "Sí") &
    (datos_emails["spam"] == "Sí")
)

print("P(Gratis y Spam) =", p_gratis_y_spam)

In [ ]:
tabla_sin_margenes = pd.crosstab(
    datos_emails["contiene_gratis"],
    datos_emails["spam"]
)

tabla_probabilidades = tabla_sin_margenes / len(datos_emails)
tabla_probabilidades

### Visualización de probabilidades conjuntas

In [ ]:
tabla_probabilidades.plot(kind="bar", figsize=(8, 4))
plt.ylabel("Probabilidad")
plt.title("Probabilidades conjuntas estimadas")
plt.xticks(rotation=0)
plt.show()

## 4. Probabilidad condicional

Este es el concepto que más vamos a usar.

La probabilidad condicional responde preguntas del tipo:

> ¿Cuál es la probabilidad de que ocurra $A$, **dado que** ya sabemos que ocurrió $B$?

Se define como:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

### Idea clave
**Condicionar cambia el universo de referencia.**

No miramos todos los casos.  
Miramos solo los casos donde ocurrió $B$.

### Ejemplo con la tabla de emails

#### 1. ¿Cuál es la probabilidad de que un email sea spam dado que contiene “gratis”?

Entre los 30 emails que contienen “gratis”, 24 son spam.

$$
P(\text{Spam}\mid \text{Gratis}) = \frac{24}{30} = 0.8
$$

#### 2. ¿Cuál es la probabilidad de que un email contenga “gratis” dado que es spam?

Entre los 40 emails spam, 24 contienen “gratis”.

$$
P(\text{Gratis}\mid \text{Spam}) = \frac{24}{40} = 0.6
$$

Estas dos probabilidades **no son iguales**.

In [ ]:
p_spam_dado_gratis = (
    np.mean(datos_emails.loc[datos_emails["contiene_gratis"] == "Sí", "spam"] == "Sí")
)

p_gratis_dado_spam = (
    np.mean(datos_emails.loc[datos_emails["spam"] == "Sí", "contiene_gratis"] == "Sí")
)

print("P(Spam | Gratis) =", p_spam_dado_gratis)
print("P(Gratis | Spam) =", p_gratis_dado_spam)

### Punto crítico

En general:

$$
P(A\mid B) \neq P(B\mid A)
$$

Ese error aparece todo el tiempo.  
Hay que tenerlo completamente claro antes de seguir con modelos probabilísticos.

### Visualización rápida

In [ ]:
comparacion = pd.Series({
    "P(Spam | Gratis)": p_spam_dado_gratis,
    "P(Gratis | Spam)": p_gratis_dado_spam
})

plt.figure(figsize=(8, 4))
plt.bar(comparacion.index, comparacion.values)
plt.ylabel("Probabilidad")
plt.title("Comparación de probabilidades condicionales")
plt.xticks(rotation=15)
plt.show()

## 5. Independencia

Dos eventos son **independientes** si conocer que ocurrió uno no cambia la probabilidad del otro.

Formalmente:

$$
P(A\mid B)=P(A)
$$

Equivalentemente:

$$
P(A\cap B)=P(A)P(B)
$$

### Ojo
Independencia **no** significa “que no pueden ocurrir juntos”.  
Eso sería otra cosa.

### Chequeo de independencia en el ejemplo

Si “ser spam” y “contener gratis” fueran independientes, debería valer:

$$
P(\text{Gratis} \cap \text{Spam}) = P(\text{Gratis})P(\text{Spam})
$$

Veamos si pasa.

In [ ]:
lhs = p_gratis_y_spam
rhs = p_gratis * p_spam

print("P(Gratis y Spam) =", lhs)
print("P(Gratis) * P(Spam) =", rhs)
print("¿Son aproximadamente iguales?", np.isclose(lhs, rhs))

No son iguales.  
Por lo tanto, en este ejemplo, las variables **no son independientes**.

Esto tiene sentido: que aparezca la palabra “gratis” cambia bastante la probabilidad de que el email sea spam.

## 6. Regla del producto

A partir de la definición de probabilidad condicional:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

si despejamos, obtenemos:

$$
P(A\cap B)=P(A\mid B)P(B)
$$

También vale:

$$
P(A\cap B)=P(B\mid A)P(A)
$$

Esta relación es muy útil porque permite conectar conjuntas con condicionales.

In [ ]:
# Verificamos la regla del producto con el ejemplo
producto_1 = p_spam_dado_gratis * p_gratis
producto_2 = p_gratis_dado_spam * p_spam

print("P(Spam | Gratis) * P(Gratis) =", producto_1)
print("P(Gratis | Spam) * P(Spam) =", producto_2)
print("P(Gratis y Spam) =", p_gratis_y_spam)

## 7. Teorema de Bayes

Si tenemos:

$$
P(A\cap B)=P(A\mid B)P(B)
$$
y también
$$
P(A\cap B)=P(B\mid A)P(A)
$$

entonces, igualando ambas expresiones:

$$
P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}
$$

Eso es el **teorema de Bayes**.

### Aplicado al ejemplo de spam

Queremos:

$$
P(\text{Spam} \mid \text{Gratis})
$$

Bayes dice:

$$
P(\text{Spam} \mid \text{Gratis}) =
\frac{P(\text{Gratis} \mid \text{Spam}) P(\text{Spam})}
{P(\text{Gratis})}
$$

Reemplazando:

$$
P(\text{Spam} \mid \text{Gratis}) =
\frac{0.6 \cdot 0.4}{0.3} = 0.8
$$

In [ ]:
bayes_spam_dado_gratis = (p_gratis_dado_spam * p_spam) / p_gratis
print("Calculado con Bayes:", bayes_spam_dado_gratis)
print("Calculado directamente:", p_spam_dado_gratis)

## 8. Conexión con machine learning

En clasificación supervisada:

- $x$: entrada,
- $y$: clase o etiqueta.

Muy frecuentemente queremos modelar:

$$
P(y \mid x)
$$

Por ejemplo:

- dado un email $x$, ¿cuál es la probabilidad de que sea spam?
- dada una imagen $x$, ¿cuál es la probabilidad de que sea un gato?
- dado un vector de atributos $x$, ¿cuál es la probabilidad de pertenecer a cierta clase?

### Modelos discriminativos
Se enfocan en modelar directamente:

$$
P(y\mid x)
$$

### Modelos generativos
Pueden modelar:

$$
P(x\mid y)
$$
junto con $P(y)$, y luego usar Bayes para obtener:

$$
P(y\mid x)=\frac{P(x\mid y)P(y)}{P(x)}
$$

Esto es exactamente la diferencia conceptual que vamos a usar al hablar de modelos generativos y discriminativos.

## 9. Ejemplo un poco más realista con datos simulados

Vamos a simular una situación simple:

- `horas_estudio`: cantidad de horas estudiadas,
- `aprueba`: si aprueba o no.

No buscamos un modelo perfecto. Solo queremos ver cómo una tabla, un gráfico y una probabilidad condicional empiezan a parecerse a un problema real de datos.

In [ ]:
n = 200
horas_estudio = rng.normal(loc=4.5, scale=2.0, size=n)
horas_estudio = np.clip(horas_estudio, 0, None)

# Probabilidad de aprobar creciente con las horas de estudio
logits = -2.0 + 0.7 * horas_estudio
prob_aprueba = 1 / (1 + np.exp(-logits))
aprueba = rng.binomial(1, prob_aprueba, size=n)

df_estudio = pd.DataFrame({
    "horas_estudio": horas_estudio,
    "aprueba": aprueba
})

df_estudio["aprueba_label"] = df_estudio["aprueba"].map({1: "Sí", 0: "No"})
df_estudio.head()

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df_estudio["horas_estudio"], bins=15)
plt.xlabel("Horas de estudio")
plt.ylabel("Frecuencia")
plt.title("Distribución de horas de estudio")
plt.show()

In [ ]:
# Convertimos horas en categorías para estimar probabilidades condicionales por grupo
bins = [0, 2, 4, 6, 8, np.inf]
labels = ["0-2", "2-4", "4-6", "6-8", "8+"]
df_estudio["rango_horas"] = pd.cut(df_estudio["horas_estudio"], bins=bins, labels=labels, right=False)

tabla_aprueba = pd.crosstab(df_estudio["rango_horas"], df_estudio["aprueba_label"], normalize="index")
tabla_aprueba

La tabla anterior estima algo del estilo:

$$
P(\text{Aprueba} \mid \text{Rango de horas})
$$

Eso ya es una probabilidad condicional calculada a partir de datos.

In [ ]:
prob_aprueba_por_rango = tabla_aprueba["Sí"]

plt.figure(figsize=(8, 4))
plt.bar(prob_aprueba_por_rango.index.astype(str), prob_aprueba_por_rango.values)
plt.xlabel("Rango de horas de estudio")
plt.ylabel("P(Aprueba | rango)")
plt.title("Probabilidad estimada de aprobar según horas de estudio")
plt.show()

### Lectura del gráfico

No hace falta que el comportamiento sea perfecto ni estrictamente monótono en cada muestra.  
La idea es que **los datos sugieren una relación probabilística** entre variables.

Este es el tipo de razonamiento que después aparece cuando entrenamos modelos sobre datos observados.

## 10. Ejemplo con una variable categórica y tablas de contingencia

Ahora armamos un ejemplo donde la variable de entrada sea categórica.

Supongamos que registramos:

- `turno`: mañana, tarde o noche,
- `entrega_tp`: si una entrega se realizó o no.

La idea no es pedagógica por el dominio, sino por la estructura de los datos.

In [ ]:
turnos = rng.choice(["Mañana", "Tarde", "Noche"], size=180, p=[0.35, 0.4, 0.25])

prob_entrega = np.where(turnos == "Mañana", 0.80,
                 np.where(turnos == "Tarde", 0.65, 0.50))

entrega = rng.binomial(1, prob_entrega)

df_turnos = pd.DataFrame({
    "turno": turnos,
    "entrega_tp": np.where(entrega == 1, "Sí", "No")
})

df_turnos.head()

In [ ]:
tabla_turnos = pd.crosstab(df_turnos["turno"], df_turnos["entrega_tp"], margins=True)
tabla_turnos

In [ ]:
tabla_turnos_cond = pd.crosstab(df_turnos["turno"], df_turnos["entrega_tp"], normalize="index")
tabla_turnos_cond

La tabla normalizada por fila estima:

$$
P(\text{Entrega} \mid \text{Turno})
$$

De nuevo: **tabla de frecuencias + normalización** = forma simple de estimar probabilidades desde datos.

In [ ]:
tabla_turnos_cond["Sí"].plot(kind="bar", figsize=(8, 4))
plt.xlabel("Turno")
plt.ylabel("P(Entrega | Turno)")
plt.title("Probabilidad estimada de entrega según turno")
plt.xticks(rotation=0)
plt.show()

## 11. Resumen conceptual

Hasta acá vimos:

### Probabilidad marginal
Mira una variable sola.

Ejemplo:
$$
P(\text{Spam})
$$

### Probabilidad conjunta
Mira dos eventos al mismo tiempo.

Ejemplo:
$$
P(\text{Gratis} \cap \text{Spam})
$$

### Probabilidad condicional
Mira la probabilidad de un evento restringiendo el universo a otro evento.

Ejemplo:
$$
P(\text{Spam} \mid \text{Gratis})
$$

### Independencia
Ocurre cuando conocer un evento no cambia la probabilidad del otro.

### Bayes
Permite reexpresar una probabilidad condicional en términos de otra.

## 12. Ejercicios propuestos

Intentá resolver estos ejercicios primero sin mirar soluciones externas.

### Ejercicio 1
En el ejemplo de emails, calculá:

1. $P(\text{No Spam})$
2. $P(\text{No Gratis})$
3. $P(\text{No Gratis} \cap \text{No Spam})$
4. $P(\text{No Spam} \mid \text{Gratis})$

### Ejercicio 2
Con `df_estudio`, calcular:

1. la probabilidad marginal de aprobar,
2. la probabilidad de estudiar al menos 6 horas,
3. la probabilidad de aprobar dado que se estudió al menos 6 horas.

### Ejercicio 3
Con `df_turnos`, responder:

1. ¿qué turno tiene mayor probabilidad estimada de entrega?
2. ¿esa diferencia implica causalidad?
3. ¿qué información adicional faltaría para hacer una afirmación causal seria?

### Ejercicio 4
Explicar por qué:

$$
P(A\mid B) \neq P(B\mid A)
$$

y construir un ejemplo propio.

## 13. Cierre

Esta notebook no intenta cubrir toda la teoría de probabilidad.  
El objetivo es:

- recuperar lenguaje básico,
- evitar errores gruesos con probabilidades condicionales,
- y dejar preparado el terreno para interpretar expresiones como:

$$
P(y\mid x)
$$

que van a aparecer una y otra vez en aprendizaje automático.

Es parte de cómo pensamos clasificación, modelado y entrenamiento.